In [0]:
# Cleansing, Modeling & Silver Layer

from pyspark.sql.functions import (
    explode, arrays_zip, col, to_timestamp, current_timestamp, row_number
)
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, TimestampType
)
from pyspark.sql.window import Window
from pyspark.sql import DataFrame
from datetime import datetime, date, timedelta

# Configuration (reuses the Bronze layer's log table -- one audit trail for both layers, distinguished by `layer_processed`)

CATALOG_VOLUME_ROOT = "/Volumes/workspace/default"

bronze_delta_path = f"{CATALOG_VOLUME_ROOT}/bronze_layer/bronze_carbon_data"
silver_delta_path = f"{CATALOG_VOLUME_ROOT}/silver_layer/silver_air_quality_readings"
log_table_path    = f"{CATALOG_VOLUME_ROOT}/bronze_layer/pipeline_execution_logs"

print("bronze_delta_path:", bronze_delta_path)
print("silver_delta_path:", silver_delta_path)
print("log_table_path   :", log_table_path)

In [0]:
# Silver Data Model (Data Dictionary) Primary key: (city, reading_time)

silver_schema = StructType([
    StructField("city", StringType(), False),
    StructField("reading_time", TimestampType(), False),
    StructField("latitude", DoubleType(), False),
    StructField("longitude", DoubleType(), False),
    StructField("elevation", DoubleType(), True),
    StructField("timezone", StringType(), True),
    StructField("carbon_monoxide_ugm3", DoubleType(), True),
    StructField("carbon_dioxide_ppm", DoubleType(), True),
    StructField("pm2_5_ugm3", DoubleType(), True),
    StructField("pm10_ugm3", DoubleType(), True),
    StructField("nitrogen_dioxide_ugm3", DoubleType(), True),
    StructField("source_file", StringType(), True),
    StructField("bronze_load_timestamp", TimestampType(), True),
    StructField("silver_load_timestamp", TimestampType(), False),
])

# Create the Silver Delta table once, empty, so MERGE INTO has a target
# to merge into on the very first run.
try:
    spark.read.format("delta").load(silver_delta_path)
    print("Silver table already exists.")
except Exception:
    empty_df = spark.createDataFrame([], silver_schema)
    empty_df.write.format("delta").save(silver_delta_path)
    print("Silver table created successfully.")

In [0]:
# Create Audit Log Table if it doesn't already exist

from pyspark.sql.types import IntegerType

log_schema = StructType([
    StructField("layer_processed", StringType(), True),
    StructField("parameter_processed", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("rows_processed", IntegerType(), True)
])

try:
    spark.read.format("delta").load(log_table_path)
    print("Log table already exists.")
except Exception:
    empty_df = spark.createDataFrame([], log_schema)
    empty_df.write.format("delta").save(log_table_path)
    print("Audit log table created successfully.")

In [0]:
# Transformation: explode, cast, clean

def clean_and_explode(bronze_df: DataFrame) -> DataFrame:
    # 1. Explode the parallel hourly arrays into one row per city-hour
    exploded = (bronze_df.select(
        "city", "latitude", "longitude", "elevation", "timezone",
        "source_file", col("load_timestamp").alias("bronze_load_timestamp"),
        explode(arrays_zip(
            col("hourly.time").alias("time"),
            col("hourly.carbon_monoxide").alias("carbon_monoxide"),
            col("hourly.carbon_dioxide").alias("carbon_dioxide"),
            col("hourly.pm2_5").alias("pm2_5"),
            col("hourly.pm10").alias("pm10"),
            col("hourly.nitrogen_dioxide").alias("nitrogen_dioxide"),
        )).alias("reading")
    ).select(
        "city", "latitude", "longitude", "elevation", "timezone",
        "source_file", "bronze_load_timestamp",
        to_timestamp(col("reading.time")).alias("reading_time"),
        col("reading.carbon_monoxide").cast("double").alias("carbon_monoxide_ugm3"),
        col("reading.carbon_dioxide").cast("double").alias("carbon_dioxide_ppm"),
        col("reading.pm2_5").cast("double").alias("pm2_5_ugm3"),
        col("reading.pm10").cast("double").alias("pm10_ugm3"),
        col("reading.nitrogen_dioxide").cast("double").alias("nitrogen_dioxide_ugm3"),
    ))

    # 2. Data quality filters: drop structurally invalid rows
    clean = exploded.filter(
        col("reading_time").isNotNull() &
        col("city").isNotNull() &
        (col("carbon_dioxide_ppm").isNull() | (col("carbon_dioxide_ppm") > 0)) &
        (col("carbon_monoxide_ugm3").isNull() | (col("carbon_monoxide_ugm3") != -999)) &
        (col("pm2_5_ugm3").isNull() | (col("pm2_5_ugm3") >= 0)) &
        (col("pm10_ugm3").isNull() | (col("pm10_ugm3") >= 0)) &
        (col("nitrogen_dioxide_ugm3").isNull() | (col("nitrogen_dioxide_ugm3") >= 0))
    )

    # 3. Deduplicate within the batch: if two overlapping Bronze loads produced the same
    #    (city, reading_time) twice, keep only the most recently-ingested
    #    Bronze row. This is also what lets a REVISED forecast value win.
    window = Window.partitionBy("city", "reading_time").orderBy(col("bronze_load_timestamp").desc())
    final = (clean
             .withColumn("rn", row_number().over(window))
             .filter(col("rn") == 1)
             .drop("rn")
             .withColumn("silver_load_timestamp", current_timestamp()))

    return final

In [0]:
# Idempotent write: MERGE INTO keyed on (city, reading_time)

from delta.tables import DeltaTable

def upsert_to_silver(final_df: DataFrame, silver_table_path: str):
    silver_table = DeltaTable.forPath(spark, silver_table_path)
    (silver_table.alias("t")
        .merge(final_df.alias("s"), "t.city = s.city AND t.reading_time = s.reading_time")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

In [0]:
# ==========================================
# Parameterized Bronze-to-Silver function (with logging)
#
# Takes a READING-DATE range (start_date, end_date) -- the dates the
# atmospheric readings themselves fall on, not when they were loaded into
# Bronze. This is what makes a backfill or a corrected re-run meaningful:
# it reprocesses Bronze data covering that window regardless of which
# Bronze load(s) it came from.


def process_bronze_to_silver(start_date: str, end_date: str,
                              bronze_table_path: str, silver_table_path: str,
                              log_table_path: str):
    layer_name = "Bronze-to-Silver"
    parameter_label = f"{start_date}_to_{end_date}"
    start_time = datetime.now()
    status = "Failed"
    row_count = 0

    print(f"--- Starting {layer_name}: {parameter_label} ---")

    try:
        bronze_df = spark.read.format("delta").load(bronze_table_path)
        final_df = clean_and_explode(bronze_df)

        # Restrict to the requested reading-date window
        final_df = final_df.filter(
            (col("reading_time") >= f"{start_date}T00:00:00") &
            (col("reading_time") <= f"{end_date}T23:59:59")
        )

        row_count = final_df.count()
        upsert_to_silver(final_df, silver_table_path)
        status = "Success"
        print(f"Success! {row_count} rows merged into Silver layer.")

    except Exception as e:
        print(f"Pipeline Failed: {e}")
        status = f"Failed: {str(e)[:400]}"

    finally:
        end_time = datetime.now()

        log_entry = spark.createDataFrame([(
            layer_name,
            parameter_label,
            start_time,
            end_time,
            status,
            row_count
        )], schema=log_schema)

        log_entry.write.format("delta").mode("append").save(log_table_path)
        print("Audit log updated.")
        print("---------------------------------")

    return status

In [0]:
# Parameterized Backfill Wrapper (Silver)

def backfill_silver(start_date: str, end_date: str,
                     bronze_table_path: str, silver_table_path: str,
                     log_table_path: str, window_days: int = 1):
    """Processes [start_date, end_date] in `window_days`-sized chunks.
    For a plain backfill, window_days=1 is fine. For the daily incremental
    job, call process_bronze_to_silver directly
    with a single 3-5 day rolling window instead of looping this."""
    current = date.fromisoformat(start_date)
    end = date.fromisoformat(end_date)
    results = []
    while current <= end:
        chunk_end = min(current + timedelta(days=window_days - 1), end)
        status = process_bronze_to_silver(
            current.isoformat(), chunk_end.isoformat(),
            bronze_table_path, silver_table_path, log_table_path
        )
        results.append((current.isoformat(), chunk_end.isoformat(), status))
        current = chunk_end + timedelta(days=1)
    return results

In [0]:
# Execute the Pipeline

try:
    spark.read.format("delta").load(silver_delta_path).limit(1).count()
except Exception:
    spark.createDataFrame([], silver_schema).write.format("delta").save(silver_delta_path)
    print("Silver table created.")

# --- Daily incremental run: rolling 5-day window
today = date.today()
window_start = (today - timedelta(days=5)).isoformat()
window_end = today.isoformat()
process_bronze_to_silver("2026-09-23", "2026-09-24",
                          bronze_delta_path, silver_delta_path, log_table_path)

# --- Example full backfill (uncomment once Bronze has the full history) ---
# backfill_silver("2023-09-01", "2026-10-03",
#                  bronze_delta_path, silver_delta_path, log_table_path,
#                  window_days=30)

# --- Idempotency proof: re-run the exact same window and confirm the
#     Silver row count does not change ---
before = spark.read.format("delta").load(silver_delta_path).count()
process_bronze_to_silver(window_start, window_end,
                          bronze_delta_path, silver_delta_path, log_table_path)
after = spark.read.format("delta").load(silver_delta_path).count()
print(f"Silver row count before second run: {before}, after: {after} (should match)")

# Verify the log
display(spark.read.format("delta").load(log_table_path).orderBy(col("start_time").desc()))

In [0]:
# Executing the full 3 year load

process_bronze_to_silver("2023-10-02", "2026-10-02",
                          bronze_delta_path, silver_delta_path, log_table_path)
display(spark.read.format("delta").load(log_table_path).orderBy(col("start_time").desc()))
